In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, chisquare
from scipy.optimize import curve_fit
import pandas as pd


In [ ]:
def gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu) ** 2 / (2 * sigma**2))

In [ ]:
phase_df = pd.read_csv("./fft_phase_analysis/10Hz/phase_analysis_results.csv")

In [ ]:
phase_df.head()

In [ ]:
camera_ids = [12574, 12606, 13251, 13703]

for id in camera_ids:

    df_data = phase_df[(phase_df["Main Camera"] == id) | (phase_df["Other Camera"] == id)]
    slope = df_data["Slope"].values
    print(len(slope))
    cnts, bin_edges = np.histogram(slope, bins=20, density=False)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2


    popt, pcov = curve_fit(gaussian, bin_centers, cnts, p0=[1, np.mean(slope), np.std(slope)])
    err = np.sqrt(np.diag(pcov))

    x_fit = np.linspace(bin_edges[0], bin_edges[-1], 1000)
    y_fit = gaussian(x_fit, *popt)


    plt.figure(figsize=(10, 6))
    plt.bar(bin_centers, cnts, width=bin_edges[1] - bin_edges[0], alpha=0.6, label="Data")
    plt.plot(x_fit, y_fit, color="red", label="Gaussian Fit")
    plt.xlabel("Phase Drift")
    plt.ylabel("Counts")
    plt.title(f"Camera {id} Histogram of Phase Drift with Gaussian Fit")
    plt.legend()
    plt.grid()

    text_str = f"A = {popt[0]:.4f} ± {err[0]:.4f}\nμ = {popt[1]:.4f} ± {err[1]:.4f}\nσ = {popt[2]:.4f} ± {err[2]:.4f}"
    plt.text(0.05, 0.95, text_str, transform=plt.gca().transAxes, fontsize=12, verticalalignment="top")
    plt.savefig(f"./fft_phase_analysis/5Hz/Camera_{id}__Phase_Drift_Fit.png")

